# Stage 4a — W2V2-L2 Linear Head
### Alertreck · Transfer Learning Paradigm

Trains a lightweight MLP classification head over **frozen wav2vec 2.0 layer-2 embeddings**
pre-extracted in Stage 2b. The 94M-parameter backbone is never touched during training;
only the head (~527K parameters) is updated.

---

## Architecture

```
Input  (768-dim L2-normalised W2V2 embedding)
  ├── Linear(768 → 512) → BN → ReLU → Dropout(0.3)
  ├── Linear(512 → 256) → BN → ReLU → Dropout(0.3)
  └── Linear(256 → 7)   →  logits
```

## Training strategy

| Phase | Epochs | Train data | LR schedule |
|-------|--------|------------|-------------|
| A | 20 | clean + aug_A (light noise, ×1) | cosine decay 3e-4 → ~0 |
| B | 15 | clean + aug_B (medium noise, ×2) | restart 3e-4 → ~0 |
| C | until patience | clean + aug_C (heavy noise, ×3) | restart 3e-4 → ~0 |

All embeddings are loaded into RAM at the start — no I/O bottleneck during training.

## Datasets required
- `alertreck-w2v2-embeddings` — output of `02b-prepare-w2v2-embeddings.ipynb`

## Cell 1 — Imports

In [ ]:
import json, math, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    roc_auc_score, roc_curve,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 2 — Configuration

In [ ]:
CFG = {
    # ── paths ── update data_root to match your Kaggle dataset mount path
    'data_root':    Path('/kaggle/input/datasets/orpheusmanga/alertreck-w2v2-embeddings/w2v2_l2'),
    'output_dir':   Path('/kaggle/working/w2v2_l2'),

    # ── model ──
    'embed_dim':    768,
    'n_classes':    7,
    'hidden_dims':  [512, 256],
    'dropout':      0.3,

    # ── training ──
    'lr':           3e-4,
    'weight_decay': 1e-4,
    'batch_size':   512,
    'epochs':       80,
    'patience':     12,
    'label_smooth': 0.1,
    'phase_epochs': {'A': 20, 'B': 15, 'C': 999},

    # ── labels ──
    'class_names': [
        'bg_animals', 'bg_wind_rain',
        'chainsaw', 'dog', 'gunshot', 'human', 'vehicle'
    ],
    'label_map': {
        'background_animals':   0,
        'background_wind_rain': 1,
        'threat_chainsaw':      2,
        'threat_dog':           3,
        'threat_gunshot':       4,
        'threat_human':         5,
        'threat_vehicle':       6,
    }
}

output_dir = CFG['output_dir']
output_dir.mkdir(parents=True, exist_ok=True)
data_root = CFG['data_root']

print(f'Data root : {data_root}')
print(f'Output    : {output_dir}')
print()

# Verify all embedding directories are present
splits = ['train', 'val', 'test', 'train_aug_A', 'train_aug_B', 'train_aug_C']
for split in splits:
    d = data_root / split
    n = len(list(d.glob('*.npz'))) if d.exists() else 0
    status = 'OK' if n > 0 else 'MISSING'
    print(f'  {split:<16} {n:>3} shards  [{status}]')

## Cell 3 — Dataset and DataLoaders

In [ ]:
class EmbeddingDataset(Dataset):
    """Loads all .npz shards from a directory into memory as tensors."""

    def __init__(self, shard_dir: Path):
        shards = sorted(Path(shard_dir).glob('*.npz'))
        if not shards:
            raise FileNotFoundError(f'No .npz shards in {shard_dir}')
        Xs, ys = [], []
        for s in shards:
            d = np.load(s)
            Xs.append(d['X'])
            ys.append(d['y'])
        self.X = torch.from_numpy(np.concatenate(Xs, axis=0)).float()
        self.y = torch.from_numpy(np.concatenate(ys, axis=0)).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def make_loader(shard_dirs, batch_size, shuffle):
    if isinstance(shard_dirs, Path):
        shard_dirs = [shard_dirs]
    datasets = [EmbeddingDataset(d) for d in shard_dirs]
    ds = datasets[0] if len(datasets) == 1 else ConcatDataset(datasets)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))


print('Loading val  embeddings into RAM…')
val_loader  = make_loader(data_root / 'val',  CFG['batch_size'], shuffle=False)
print(f'  val  : {len(val_loader.dataset):,} embeddings')

print('Loading test embeddings into RAM…')
test_loader = make_loader(data_root / 'test', CFG['batch_size'], shuffle=False)
print(f'  test : {len(test_loader.dataset):,} embeddings')

In [ ]:
# Diagnostic — check class distribution in each split
from collections import Counter

for split_name, loader in [('val', val_loader), ('test', test_loader)]:
    y_all = loader.dataset.dataset.y if hasattr(loader.dataset, 'dataset') else loader.dataset.y
    counts = Counter(y_all.numpy())
    total  = len(y_all)
    print(f'{split_name} — {total} embeddings')
    for cls_id in sorted(counts):
        name = CFG['class_names'][cls_id]
        n    = counts[cls_id]
        print(f'  {name:<22}  {n:>4}  ({100*n/total:5.1f}%)')
    print()

# Also check train clean split
train_clean = EmbeddingDataset(data_root / 'train')
counts_tr   = Counter(train_clean.y.numpy())
total_tr    = len(train_clean.y)
print(f'train (clean) — {total_tr} embeddings')
for cls_id in sorted(counts_tr):
    name = CFG['class_names'][cls_id]
    n    = counts_tr[cls_id]
    print(f'  {name:<22}  {n:>4}  ({100*n/total_tr:5.1f}%)')

## Cell 4 — Model

In [ ]:
class W2V2Head(nn.Module):
    """Lightweight MLP classification head over frozen W2V2 embeddings."""

    def __init__(self, embed_dim=768, hidden_dims=None, n_classes=7, dropout=0.3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256]
        layers = []
        in_dim = embed_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = W2V2Head(
    embed_dim=CFG['embed_dim'],
    hidden_dims=CFG['hidden_dims'],
    n_classes=CFG['n_classes'],
    dropout=CFG['dropout'],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'W2V2Head parameters : {n_params:,}')
print(model)

## Cell 5 — Training loop

In [ ]:
PHASE_ORDER = ['A', 'B', 'C']
PHASE_CAP   = CFG['phase_epochs']
max_epochs  = CFG['epochs']

criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smooth'])
optimizer = optim.Adam(model.parameters(), lr=CFG['lr'],
                       weight_decay=CFG['weight_decay'])

history = {
    'train_loss': [], 'val_loss'  : [],
    'train_acc' : [], 'val_acc'   : [],
    'lr'        : [], 'phase'     : [],
}

best_val_loss    = float('inf')
best_epoch       = 0
best_phase       = 'A'
patience_counter = 0
phase_idx        = 0
epochs_in_phase  = 0
current_phase    = PHASE_ORDER[phase_idx]


def load_train(phase):
    return make_loader(
        [data_root / 'train', data_root / f'train_aug_{phase}'],
        CFG['batch_size'], shuffle=True
    )


train_loader = load_train(current_phase)
print(f'Phase A — {len(train_loader.dataset):,} train embeddings')
print()

t0 = time.time()
for epoch in range(1, max_epochs + 1):

    # ── phase transition ──────────────────────────────────────────────────
    if epochs_in_phase >= PHASE_CAP[current_phase] and phase_idx < len(PHASE_ORDER) - 1:
        phase_idx       += 1
        current_phase    = PHASE_ORDER[phase_idx]
        epochs_in_phase  = 0
        patience_counter = 0
        train_loader     = load_train(current_phase)
        for pg in optimizer.param_groups:
            pg['lr'] = CFG['lr']
        print(f'  → Phase {current_phase}  ({len(train_loader.dataset):,} train embeddings)')
        print()

    # ── cosine LR within phase ────────────────────────────────────────────
    phase_len = PHASE_CAP[current_phase] if current_phase != 'C' else max_epochs
    progress  = epochs_in_phase / max(phase_len, 1)
    lr = CFG['lr'] * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    # ── train ─────────────────────────────────────────────────────────────
    model.train()
    tr_loss = tr_correct = tr_total = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += (logits.argmax(1) == y).sum().item()
        tr_total   += len(y)
    tr_loss /= tr_total
    tr_acc   = tr_correct / tr_total

    # ── validate ──────────────────────────────────────────────────────────
    model.eval()
    va_loss = va_correct = va_total = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            logits   = model(X)
            loss     = criterion(logits, y)
            va_loss    += loss.item() * len(y)
            va_correct += (logits.argmax(1) == y).sum().item()
            va_total   += len(y)
    va_loss /= va_total
    va_acc   = va_correct / va_total

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    history['lr'].append(lr)
    history['phase'].append(current_phase)

    epochs_in_phase += 1

    # ── checkpoint ────────────────────────────────────────────────────────
    tag = ''
    if va_loss < best_val_loss:
        best_val_loss    = va_loss
        best_epoch       = epoch
        best_phase       = current_phase
        patience_counter = 0
        cfg_safe = {k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()}
        torch.save({'epoch': epoch, 'phase': current_phase,
                    'model_state': model.state_dict(),
                    'optim_state': optimizer.state_dict(),
                    'val_loss': va_loss, 'val_acc': va_acc,
                    'cfg': cfg_safe},
                   output_dir / 'best_model.pt')
        tag = ' <- saved'
    else:
        patience_counter += 1

    print(f'Ep {epoch:03d}/{max_epochs} [{current_phase}]  '
          f'tr_loss={tr_loss:.6f}  tr_acc={tr_acc:.4f}  '
          f'va_loss={va_loss:.6f}  va_acc={va_acc:.4f}  '
          f'lr={lr:.2e}{tag}')

    if patience_counter >= CFG['patience'] and current_phase == 'C':
        print(f'\nEarly stopping at epoch {epoch}')
        break

elapsed = time.time() - t0
print(f'\nDone in {elapsed / 60:.1f} min')
print(f'Best epoch {best_epoch} | phase {best_phase} | val_loss {best_val_loss:.6f}')

## Cell 6 — Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = list(range(1, len(history['train_loss']) + 1))

# Shade phase regions
phase_colors = {'A': '#e3f2fd', 'B': '#fff9c4', 'C': '#e8f5e9'}
ph_starts = {}
for i, ph in enumerate(history['phase']):
    if ph not in ph_starts:
        ph_starts[ph] = i
ph_order = sorted(ph_starts, key=lambda p: ph_starts[p])
for idx, ph in enumerate(ph_order):
    start = ph_starts[ph] + 1
    end   = ph_starts[ph_order[idx + 1]] + 1 if idx + 1 < len(ph_order) else epochs_x[-1] + 1
    for ax in axes:
        ax.axvspan(start, end, alpha=0.15, color=phase_colors[ph], label=f'Phase {ph}')

axes[0].plot(epochs_x, history['train_loss'], label='Train', color='#1565C0', lw=1.5)
axes[0].plot(epochs_x, history['val_loss'],   label='Val',   color='#C62828', lw=1.5)
axes[0].axvline(best_epoch, color='gray', ls='--', alpha=0.7, label=f'Best ep {best_epoch}')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, history['train_acc'], label='Train', color='#1565C0', lw=1.5)
axes[1].plot(epochs_x, history['val_acc'],   label='Val',   color='#C62828', lw=1.5)
axes[1].axvline(best_epoch, color='gray', ls='--', alpha=0.7, label=f'Best ep {best_epoch}')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('W2V2-L2 Head — Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> training_curves.png')

## Cell 7 — Load best model and evaluate on test set

In [ ]:
import pathlib
torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.Path])
ckpt = torch.load(output_dir / 'best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best model')
print(f'  epoch    : {ckpt["epoch"]}')
print(f'  phase    : {ckpt["phase"]}')
print(f'  val_loss : {ckpt["val_loss"]:.6f}')
print(f'  val_acc  : {ckpt["val_acc"]:.4f}')


def evaluate(loader):
    all_logits, all_y = [], []
    with torch.no_grad():
        for X, y in loader:
            all_logits.append(model(X.to(device)).cpu())
            all_y.append(y)
    logits = torch.cat(all_logits)
    y_true = torch.cat(all_y).numpy()
    probs  = torch.softmax(logits, dim=1).numpy()
    preds  = logits.argmax(1).numpy()
    return y_true, preds, probs


y_true, y_pred, y_prob = evaluate(test_loader)

test_acc      = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average='macro')
macro_auc     = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')

print()
print('=== Test Results ===')
print(f'Accuracy  : {test_acc:.4f}')
print(f'Macro F1  : {test_macro_f1:.4f}')
print(f'Macro AUC : {macro_auc:.4f}')

## Cell 8 — Confusion matrix

In [ ]:
class_names = CFG['class_names']
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Counts', 'Row-normalised']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar=True)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('W2V2-L2 — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> confusion_matrix.png')

## Cell 9 — Per-class metrics

In [ ]:
per_class_f1  = f1_score(y_true, y_pred, average=None)
per_class_auc = {}
for i, name in enumerate(class_names):
    binary_y = (y_true == i).astype(int)
    per_class_auc[name] = roc_auc_score(binary_y, y_prob[:, i])

print(f'{"Class":<22}  {"F1":>6}  {"AUC":>6}')
print('-' * 40)
for i, name in enumerate(class_names):
    print(f'{name:<22}  {per_class_f1[i]:>6.4f}  {per_class_auc[name]:>6.4f}')
print('-' * 40)
print(f'{"macro":<22}  {test_macro_f1:>6.4f}  {macro_auc:>6.4f}')

## Cell 10 — Per-class ROC curves

In [ ]:
n_cols = 4
n_rows = math.ceil(CFG['n_classes'] / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.flatten()
colors = plt.cm.tab10.colors

for i, name in enumerate(class_names):
    binary_y = (y_true == i).astype(int)
    fpr, tpr, _ = roc_curve(binary_y, y_prob[:, i])
    auc = per_class_auc[name]
    axes[i].plot(fpr, tpr, color=colors[i], lw=2, label=f'AUC={auc:.3f}')
    axes[i].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[i].set_title(name, fontsize=9)
    axes[i].set_xlabel('FPR', fontsize=8)
    axes[i].set_ylabel('TPR', fontsize=8)
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('W2V2-L2 — Per-class ROC Curves (Test Set)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> roc_curves.png')

## Cell 11 — ONNX export

In [ ]:
onnx_path = output_dir / 'w2v2_head.onnx'
dummy = torch.randn(1, CFG['embed_dim'], device=device)

model.eval()
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['embedding'],
    output_names=['logits'],
    dynamic_axes={'embedding': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
    dynamo=False,
)
size_kb = onnx_path.stat().st_size / 1024
print(f'ONNX exported -> {onnx_path}  ({size_kb:.1f} KB)')

## Cell 12 — Save results

In [ ]:
results = {
    'model':          'W2V2Head',
    'backbone':       'facebook/wav2vec2-base',
    'w2v2_layer':     2,
    'embed_dim':      CFG['embed_dim'],
    'hidden_dims':    CFG['hidden_dims'],
    'n_params_head':  n_params,
    'best_epoch':     best_epoch,
    'best_phase':     best_phase,
    'best_val_loss':  best_val_loss,
    'test_acc':       round(float(test_acc),      6),
    'test_macro_f1':  round(float(test_macro_f1), 6),
    'macro_auc':      round(float(macro_auc),     6),
    'per_class_f1':   {name: round(float(f), 6) for name, f in zip(class_names, per_class_f1)},
    'per_class_auc':  {k: round(v, 6) for k, v in per_class_auc.items()},
    'history': {
        'train_loss': history['train_loss'],
        'val_loss'  : history['val_loss'],
        'train_acc' : history['train_acc'],
        'val_acc'   : history['val_acc'],
        'lr'        : history['lr'],
        'phase'     : history['phase'],
    },
}

results_path = output_dir / 'results.json'
results_path.write_text(json.dumps(results, indent=2))
print(f'Results saved -> {results_path}')
print()
print(f'Model     : W2V2-L2 (frozen backbone + MLP head)')
print(f'Accuracy  : {test_acc:.4f}')
print(f'Macro F1  : {test_macro_f1:.4f}')
print(f'Macro AUC : {macro_auc:.4f}')